<a href="https://colab.research.google.com/github/Krishna28Gupta/Health_Risk_Predictor/blob/main/Model_comparsion_stroke_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

In [3]:
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())
df['avg_glucose_level'] = df['avg_glucose_level'].fillna(df['avg_glucose_level'].mean())

In [4]:
df['smoking_status'] = df['smoking_status'].fillna(df['smoking_status'].mode()[0])

In [5]:
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

In [6]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [7]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [8]:
X = df.drop(['id', 'stroke'], axis=1)
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_resampled.value_counts())

Before SMOTE: stroke
0    3889
1     199
Name: count, dtype: int64
After SMOTE: stroke
0    3889
1    3889
Name: count, dtype: int64


In [10]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

In [11]:
results={}

In [12]:
for name,model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    results[name] = accuracy
    print(f"\n🔹 {name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, y_pred))


🔹 Logistic Regression
Accuracy: 0.7847
              precision    recall  f1-score   support

           0       0.98      0.79      0.87       972
           1       0.15      0.72      0.25        50

    accuracy                           0.78      1022
   macro avg       0.57      0.75      0.56      1022
weighted avg       0.94      0.78      0.84      1022


🔹 Decision Tree
Accuracy: 0.8640
              precision    recall  f1-score   support

           0       0.96      0.90      0.93       972
           1       0.11      0.24      0.15        50

    accuracy                           0.86      1022
   macro avg       0.53      0.57      0.54      1022
weighted avg       0.92      0.86      0.89      1022


🔹 Random Forest
Accuracy: 0.9022
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       972
           1       0.12      0.16      0.14        50

    accuracy                           0.90      1022
   macro avg       0

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:37:37] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🔹 XGBoost
Accuracy: 0.8992
              precision    recall  f1-score   support

           0       0.95      0.94      0.95       972
           1       0.10      0.14      0.12        50

    accuracy                           0.90      1022
   macro avg       0.53      0.54      0.53      1022
weighted avg       0.91      0.90      0.91      1022



In [13]:
print("\n===== Model Comparison =====")
for model, acc in results.items():
    print(f"{model}: {acc:.4f}")


===== Model Comparison =====
Logistic Regression: 0.7847
Decision Tree: 0.8640
Random Forest: 0.9022
KNN: 0.8249
SVM: 0.7065
XGBoost: 0.8992
